# 4. Calendrier : repos, charge récente, coupes d'Europe

**Question** : le repos et la charge récente sont-ils liés au nombre de buts, et sur quelle part des lignes ces variables sont-elles fiables ?

> Notebook **descriptif** (partie 3, ADR-0034). Il ne sert ni à choisir une variable ni à régler un paramètre :
> la sélection appartient au protocole de la partie 4 (validation glissante, ablation). Corrélation n'est pas apport.
> Données : période de développement seulement (matchs antérieurs au 1er juillet 2025, scellé de l'ADR-0012),
> lues par la porte `foot_predictor.features.sources`. Sorties effacées au commit (`nbstripout`) : réexécuter pour les voir.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Seule porte vers les données (ADR-0028) : jeu versionné et, au besoin, matchs de la période de développement.
from foot_predictor.features.sources import load_dataset, load_matches

pd.set_option("display.width", 140)
NAMES = {39: "PL", 40: "Champ.", 140: "Liga", 141: "Segunda", 78: "BL", 79: "2. BL",
         135: "Serie A", 136: "Serie B", 61: "L1", 62: "L2"}
TOP5 = [39, 140, 78, 135, 61]
frame, manifest = load_dataset()
print("Jeu de données :", manifest["version"], "-", len(frame), "lignes, jusqu'au", frame["match_day"].max())
home = frame[frame["is_home"]].copy()  # une ligne par match (celle de l'équipe à domicile)
home["total"] = home["goals_for"] + home["goals_against"]

## Part des lignes fiables (toutes les coupes du pays dans `staging`), par saison

In [ ]:
display((frame.groupby("season_year")["rest_reliable"].mean() * 100).round(1).to_frame("% fiables").T)

## Repos et buts (lignes fiables, top 5)

In [ ]:
sub = frame[frame["rest_reliable"] & frame["eval_population"]].copy()
sub["repos"] = pd.cut(sub["rest_days"], [0, 3, 4, 5, 6, 7, 10, 400], right=True)
display(sub.groupby("repos", observed=True).agg(lignes=("goals_for", "size"), buts_pour=("goals_for", "mean"),
                                               buts_contre=("goals_against", "mean")).round(3))
sub["ecart_repos"] = sub["rest_days"] - sub["opp_rest_days"]
sub["ecart"] = pd.cut(sub["ecart_repos"], [-400, -3, -1, 0, 2, 400])
display(sub.groupby("ecart", observed=True).agg(lignes=("goals_for", "size"), buts_pour=("goals_for", "mean")).round(3))

## Charge sur 14 jours et match européen récent

In [ ]:
display(sub.groupby("matches_last_14d").agg(lignes=("goals_for", "size"), buts_pour=("goals_for", "mean"),
                                           buts_contre=("goals_against", "mean")).round(3))
display(sub.groupby("european_match_last_4d").agg(lignes=("goals_for", "size"), buts_pour=("goals_for", "mean"),
                                                 buts_contre=("goals_against", "mean"), elo=("elo_pre", "mean")).round(3))

## Conclusion

- **Fiabilité** : le repos n'est fiable (toutes les coupes du pays chargées) que depuis 2015-16 (58 % des lignes), puis 79 % en 2016-17 et 2017-18, et 100 % à partir de 2018-19.
- **Liens observés** (lignes fiables du top 5) : un repos court (au plus 3 jours) va avec plus de buts marqués (1,53 contre 1,34 après 6 ou 7 jours de repos) et moins de buts encaissés. Un match européen dans les 4 jours va avec 1,75 but marqué contre 1,36, mais l'Elo moyen de ces équipes vaut 1 666 contre 1 534.
- **Lecture** : ces liens reflètent surtout la *force* des équipes qui jouent les coupes, pas un effet de la fatigue (confusion). Seule une ablation, à Elo et glissants donnés (partie 4), dira si le groupe G3 apporte quelque chose. Rappel : G3 n'est disponible qu'en rejeu (ADR-0011).